# Agentry: Predictive Runtime Control Layer for Autonomous Agent Fleets
### Powered by Prior Labs TabPFN-3.5 Tabular Foundation Model & Edge Intelligence

**Prior Labs TabPFN-3.5 Global Hackathon 2026 Submission**

Agentry is a predictive runtime control layer for multi-agent fleets. It monitors real-time tabular execution telemetry (token velocities, repetition entropy, error streaks, tool latencies, and thought traces) and predicts unrecoverable failure cascades (**Infinite Loops**, **Context / Cost Runaways**, and **Tool Hallucinations**) before compute is incinerated.

## 1. Import Dependencies & Initialize Agentry

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from agentry.telemetry import load_telemetry_data, AgentStepTelemetry
from agentry.engine import TabPFNGuardrailEngine
from agentry.agent import AgentrySentry
from agentry.benchmark import GuardrailBenchmarkSuite
from agentry.guard import AgentryGuard

print("Agentry modules loaded successfully!")

## 2. Load Real-World SWE-bench Agent Telemetry
We evaluate on genuine agent execution traces streamed from `nebius/SWE-agent-trajectories` (1,156 steps across 55 unique developer sessions).


In [ ]:
df = load_telemetry_data()
print(f"Total telemetry steps: {len(df)}")
print(f"Unique agent sessions: {df['session_id'].nunique()}")
print("\nFailure Mode Distribution:")
print(df['failure_status'].value_counts())
df[['session_id', 'step_index', 'tool_name', 'error_streak', 'repetition_score', 'accumulated_cost_usd', 'failure_status']].head(10)

## 3. Fit TabPFN-3.5 Guardrail Engine
TabPFN utilizes In-Context Learning (ICL) directly across multimodal tabular features: sequential grouped session dynamics (`group_col='session_id'`, `group_time_col='step_index'`), numerical metrics, and raw thought traces.

In [ ]:
engine = TabPFNGuardrailEngine()
engine.fit(df)
print(f"Engine fitted successfully. Cloud TabPFN active: {engine.is_cloud_tabpfn}")

## 4. Real-Time Step Risk Evaluation & Unified Economic Policy
Simulate a live agent step and observe the real-time TabPFN risk probability, predicted failure mode, cost projection, and Sentry intervention.

In [ ]:
sentry = AgentrySentry(engine)

# Test step: Repetitive tool loop with an error streak
risky_step = AgentStepTelemetry(
    session_id="demo_run_01",
    step_index=4,
    agent_role="SWE-Coder",
    model_name="swe-agent-llama-70b",
    tool_name="bash",
    step_latency_ms=2400.0,
    prompt_tokens=18500,
    completion_tokens=2200,
    total_tokens=20700,
    tool_call_count=5,
    error_streak=3,
    repetition_score=0.82,
    thought_length=140,
    accumulated_cost_usd=0.062,
    thought_trace="Executing git checkout again after syntax error...",
    failure_status="INFINITE_LOOP",
    is_failure=1,
    final_cost_usd=0.45
)

decision = sentry.audit_step(risky_step)

print("=== SENTRY AUDIT RESULT ===")
print(f"Action:                  {decision.action}")
print(f"Risk Level:              {decision.risk_level}")
print(f"Failure Probability:     {decision.tabpfn_assessment.failure_probability * 100:.1f}%")
print(f"Predicted Failure Mode:  {decision.tabpfn_assessment.predicted_failure_mode}")
print(f"Projected Cost:          ${decision.tabpfn_assessment.projected_final_cost_usd:.4f}")
print(f"Estimated Cost Saved:    ${decision.estimated_cost_saved_usd:.4f}")
print(f"Sentry Reason:           {decision.reason}")

## 5. Rigorous Empirical Benchmark (Unseen Trajectory Group Split)
Compare Failure Recall, False-Stop Rate (FPR), ROC-AUC, and Cost MAE across models on **unseen agent sessions**.

In [ ]:
suite = GuardrailBenchmarkSuite()
results = suite.run_benchmark(group_split=True)

records = [
    {
        "Model Architecture": r.model_name,
        "Unseen Test Sessions": f"{r.test_sessions_count} sessions",
        "Balanced Acc (%)": f"{r.classification_balanced_acc * 100:.1f}%",
        "F1 Macro (%)": f"{r.classification_f1_macro * 100:.1f}%",
        "ROC-AUC": f"{r.classification_roc_auc:.3f}",
        "Failure Recall": f"{r.failure_recall * 100:.1f}%",
        "False-Stop Rate": f"{r.false_stop_rate * 100:.1f}%",
        "Cost MAE ($)": f"${r.regression_mae_usd:.4f}",
        "Cost R2": f"{r.regression_r2:.3f}",
        "Inf Latency": f"{r.inference_latency_ms:.2f} ms"
    }
    for r in results
]
bench_df = pd.DataFrame(records)
bench_df

## 6. Equal-Success-Rate Fleet Runtime Experiment
Evaluating the fundamental trade-off: **Preserving successful tasks vs Slashing token burn.**

In [ ]:
from scripts.run_equal_success_experiment import run_fleet_simulation
regimes, total_succ, total_fail = run_fleet_simulation()

baseline_tokens = regimes['Unprotected Fleet (No Guard)']['total_tokens']
fleet_records = []
for name, stats in regimes.items():
    succ_pct = (stats['completed_successes'] / max(1, total_succ)) * 100
    reduc = ((baseline_tokens - stats['total_tokens']) / baseline_tokens) * 100
    fleet_records.append({
        'Strategy': name,
        'Task Success Rate': f'{succ_pct:.1f}% ({stats["completed_successes"]}/{total_succ})',
        'False Kills': stats['false_kills'],
        'Runaways Caught': f'{stats["runaways_interrupted"]}/{total_fail}',
        'Total Tokens': f'{stats["total_tokens"]:,}',
        'Fleet Cost': f'${stats["total_cost_usd"]:.4f}',
        'Compute Reduction': 'Baseline (0%)' if reduc == 0 else f'-{reduc:.1f}%'
    })
pd.DataFrame(fleet_records)

## 7. Human-in-the-Loop (HITL) Supervisor Control & Hot Directives
When an agent enters high-risk or ambiguous states, Agentry escalates execution to human supervisors via `hitl_gateway`. Operators can inspect telemetry, inject corrective directives, or approve resumes.

In [ ]:
from agentry.hitl import hitl_gateway

# Create an escalation request from a high-risk Sentry decision
escalation = hitl_gateway.create_escalation(decision, tool_name="bash")
print(f"Escalated Request ID: {escalation.request_id}")
print(f"Initial Status:        {escalation.status}")
print(f"Failure Probability:   {escalation.failure_probability * 100:.1f}%")

# Operator inspects pending queue and injects corrective steering directive
pending = hitl_gateway.list_requests(status="PENDING")
print(f"Pending Requests:      {len(pending)}")

resolved = hitl_gateway.resolve(
    request_id=escalation.request_id,
    resolution="REROUTE",
    comment="Operator override: syntax error in bash command, steering to alternative tool.",
    custom_directive="OPERATOR DIRECTIVE: Do not rerun git checkout. Inspect git status and file tree first."
)
print(f"\nResolved Status:       {resolved.status}")
print(f"Injected Directive:    {resolved.custom_directive}")

## 8. Automated Forensic Incident Post-Mortem & Audit Reporting
Agentry logs all decisions to an ACID SQLite WAL audit database and can generate comprehensive AI safety compliance reports in Markdown and HTML.

In [ ]:
from agentry.report import generate_incident_report
from agentry.storage import AuditStorage

# Record decision to persistent storage
storage = AuditStorage()
storage.record_decision(decision)

# Generate audit-ready Markdown incident report
report_md = generate_incident_report(session_id=decision.session_id, format="markdown")
print("\n".join(report_md.splitlines()[:25]))
print("\n... [Report truncated for display. Full forensic metrics, trace history, and recommendations included.]")

## 9. Enterprise Zero-Code-Change OpenAI Reverse Proxy Architecture
Agentry provides an OpenAI-compatible reverse proxy (`http://localhost:8787/v1`). Any existing agent (LangChain, AutoGen, CrewAI, or raw OpenAI SDK) can be guarded simply by pointing its `base_url` to Agentry.

In [ ]:
# Example: Protecting an agent with 1 line of code change
code_example = '''
from openai import OpenAI

# Point to Agentry proxy with TabPFN runtime protection
client = OpenAI(
    base_url="http://localhost:8787/v1",
    api_key="your-openai-or-mock-key"
)

# Normal chat completion call - Agentry extracts tabular telemetry in real-time
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Refactor database migration script"}],
    extra_headers={"X-Session-ID": "session_dev_42", "X-Agent-Role": "SWE-Worker"}
)
'''
print(code_example.strip())